# Periodograms of a sampled sine wave

We sample a noiseless sine wave, then recover its period with (1) the discrete Fourier transform and (2) the Lomb-Scargle periodogram.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

In [ ]:
# Generate some fake sine wave data
amp = 0.231
P = 44.319
phi = 1.761

tstart = 105.751
tend = 385.182
tstep = P / 30.761
t = np.arange(tstart, tend, tstep)
y = amp * np.sin(2.0 * np.pi * t / P + phi)
tlen = len(t)

fig, ax = plt.subplots()
ax.scatter(t, y, marker="o", color="r")
ax.set_xlabel("time [d]")
ax.set_ylabel("y")
plt.show()

## Discrete Fourier transform

Element $s$ of the DFT corresponds to a period of $(t_\mathrm{end}-t_\mathrm{start})/s$; element 0 is the zero-frequency (mean) term.

In [ ]:
fft = np.abs(np.fft.fft(y))

# Period at each DFT element (skipping s = 0, which has infinite period)
s = np.arange(1, tlen)
Ptab = (tend - tstart) / s
fft = fft[1:]

In [ ]:
fig, ax = plt.subplots()
ax.plot(Ptab, fft, color="k")
ax.scatter(Ptab, fft, color="k")
ax.set_xlim(tstep, tend - tstart)
ax.set_xlabel("period [d]")
ax.set_ylabel("|DFT|")
plt.show()
print(f"Peak power occurs at {Ptab[np.argmax(fft)]:.2f} days")
print(f"(true period = {P} days)")

## Lomb-Scargle periodogram

`scipy.signal.lombscargle` takes *angular* frequencies. We use `normalize="amplitude"` so the peak height recovers the sine amplitude (this normalisation requires SciPy ≥ 1.15).

In [ ]:
# Define a frequency array to test
nfreq = 1000
freqmax = np.pi / tstep
freqmin = 2.0 * np.pi / (tend - tstart)
freqs = np.linspace(freqmin, freqmax, nfreq, endpoint=False)
periods = 2.0 * np.pi / freqs

LSpower = signal.lombscargle(t, y, freqs, precenter=True)
LSamp = np.sqrt(4.0 * LSpower / tlen)

In [ ]:
fig, ax = plt.subplots()
ax.plot(periods, LSamp, color="k")
ax.scatter(periods, LSamp, color="k", s=8)
ax.set_xlim(tstep, tend - tstart)
ax.set_xlabel("period [d]")
ax.set_ylabel("Amp")
plt.show()
print(f"Peak power occurs at {periods[np.argmax(LSamp)]:.2f} days")
print(f"(true period = {P} days)")